# Démo : croissance du contexte

**Contexte :** assistant du service client de Lagune Banque.

L'historique est renvoyé au modèle à chaque appel : plus la conversation avance, plus on envoie de tokens. Cette démo compare trois façons de le gérer sur une conversation de neuf tours : tout garder, supprimer les anciens messages (troncature) ou les remplacer par un résumé. Référence : [mémoire à court terme](https://docs.langchain.com/oss/python/langchain/short-term-memory).

**Sommaire**
- [Configuration](#configuration)
- [Conversation](#conversation)
- [Trois stratégies](#trois-strategies)
- [Tokens envoyés à chaque tour](#tokens-envoyes-a-chaque-tour)
- [Dernier tour : ce que le modèle reçoit](#dernier-tour)
- [Bilan](#bilan)

## <a id="configuration"></a>Configuration

In [1]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path("."), Path(".."), Path("../.."), Path("../../.."), Path("../../../.."), Path("starter"), Path("../../starter")] if (p / "code_partage").exists()).resolve()
sys.path[:0] = [str(RACINE / "code_partage"), str(RACINE / "03_agents" / "socle")]

import re

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

from base import new_model
from config import prepare

In [2]:
prepare()

Connexion OK : gemini/gemini-3.5-flash-lite


## <a id="conversation"></a>Conversation

Le client donne sa référence de prêt, la date et le montant de son remboursement dans les trois premiers messages. Le dernier message est la **question de contrôle** : elle demande de les rappeler. Si l'assistant les retrouve, le début de la conversation lui est encore accessible.

In [3]:
SYSTEM = ("Tu es l'assistant du service client de Lagune Banque, un programme et non une personne. Tu n'as accès à aucun système : "
          "tu recueilles les informations du client et tu expliques la marche à suivre en trois phrases au plus. Tu ne devines rien.")

TOURS = [
    "Mon numéro de prêt est PRT-0002.",
    "J'ai remboursé hier, le 15 novembre.",
    "31 500 FCFA.",
    "Le paiement est passé par Lagune Money et j'ai reçu un SMS de confirmation.",
    "Depuis, mon accès à un nouveau prêt est suspendu. C'est normal ?",
    "Est-ce que des frais de retard vont m'être facturés ?",
    "Combien de temps faut-il pour régulariser ?",
    "Et si le conseiller refuse ma demande ?",
    "Rappelle-moi en une phrase la référence du prêt, la date du remboursement et le montant que je t'ai donnés.",
]


def missing_elements(reponse):
    motifs = {"prêt": r"PRT-0002", "date": r"15 novembre|2026-11-15|15/11", "montant": r"31[\s\u00a0\u202f.]?500"}
    return [nom for nom, motif in motifs.items() if not re.search(motif, reponse, re.IGNORECASE)]

## <a id="trois-strategies"></a>Trois stratégies

Une règle commune : dès que l'historique atteint `TRIGGER` messages, on ne garde que les `KEEP` derniers. Seul le sort des anciens messages change.

- **Complet** : on ne fait rien, tout est renvoyé.
- **Troncature** : les anciens messages sont supprimés.
- **Résumé** : les anciens messages sont remplacés par un résumé produit par le modèle, qui s'ajoute au system prompt.

In [4]:
TRIGGER, KEEP = 9, 3

model = new_model()
CONSIGNE_RESUME = ("Résume cet échange entre un client et l'assistant de Lagune Banque en quelques lignes. "
                   "Conserve tous les identifiants, dates et montants cités.\n\n{contenu}")


def full(history, memo):
    return history


def truncation(history, memo):
    if len(history) >= TRIGGER:
        del history[:-KEEP]
    return history


def summary(history, memo):
    if len(history) >= TRIGGER:
        anciens = "\n".join(f"{'Client' if isinstance(m, HumanMessage) else 'Assistant'} : {m.content}" for m in history[:-KEEP])
        contenu = (f"Résumé précédent : {memo['text']}\n\n" if memo["text"] else "") + anciens
        reponse = model.invoke(CONSIGNE_RESUME.format(contenu=contenu))
        memo["text"] = str(reponse.content)
        memo["tokens"].append(reponse.usage_metadata["input_tokens"])
        del history[:-KEEP]
    return history

`converse` rejoue la conversation avec une stratégie. À chaque tour, le modèle reçoit le system prompt (complété du résumé s'il existe) et les messages que la stratégie a gardés. `usage_metadata` donne les tokens d'entrée de l'appel.

In [5]:
def converse(strategy):
    history, memo, tokens = [], {"text": "", "tokens": []}, []
    for texte in TOURS:
        history.append(HumanMessage(texte))
        kept = list(strategy(history, memo))
        consigne = SYSTEM + (f"\nRésumé de l'échange précédent : {memo['text']}" if memo["text"] else "")
        reponse = model.invoke([SystemMessage(consigne)] + kept)
        history.append(AIMessage(str(reponse.content)))
        tokens.append(reponse.usage_metadata["input_tokens"])
    return {"tokens": tokens, "envoyes": kept, "resume": memo["text"], "reponse": str(reponse.content), "resume_tokens": sum(memo["tokens"])}


RESULTATS = {"complet": converse(full), "troncature": converse(truncation), "résumé": converse(summary)}

## <a id="tokens-envoyes-a-chaque-tour"></a>Tokens envoyés à chaque tour

Chaque barre est le nombre de tokens d'entrée envoyés au modèle à ce tour (hors appels de résumé).

In [6]:
maximum = max(max(run["tokens"]) for run in RESULTATS.values())
for nom, run in RESULTATS.items():
    print(nom)
    for tour, tokens in enumerate(run["tokens"], 1):
        print(f"  tour {tour}  {'█' * round(40 * tokens / maximum):<40} {tokens}")
    print()

complet
  tour 1  ████                                     70
  tour 2  ████████                                 143
  tour 3  ████████████                             216
  tour 4  █████████████████                        311
  tour 5  ██████████████████████                   408
  tour 6  ███████████████████████████              492
  tour 7  ███████████████████████████████          569
  tour 8  ███████████████████████████████████      636
  tour 9  ████████████████████████████████████████ 726

troncature
  tour 1  ████                                     70
  tour 2  ███████                                  131
  tour 3  ████████████                             224
  tour 4  ███████████████████                      339
  tour 5  █████████                                166
  tour 6  ██████████████                           247
  tour 7  █████████████████                        315
  tour 8  ████████                                 137
  tour 9  ████████████                         

## <a id="dernier-tour"></a>Dernier tour : ce que le modèle reçoit

Pour chaque stratégie, les messages envoyés au tour de la question de contrôle, la réponse et les éléments qui manquent.

In [7]:
def short(texte, n=70):
    return texte if len(texte) <= n else texte[:n] + "…"


for nom, run in RESULTATS.items():
    print(f"== {nom} ==")
    if run["resume"]:
        print(f"  [résumé] {short(run['resume'], 110)}")
    for m in run["envoyes"]:
        print(f"  {'Client' if isinstance(m, HumanMessage) else 'Assistant':9s}: {short(str(m.content))}")
    print(f"  Réponse : {run['reponse']}")
    print(f"  Manquants : {missing_elements(run['reponse'])}\n")

== complet ==
  Client   : Mon numéro de prêt est PRT-0002.
  Assistant: Bonjour, je prends note de votre numéro de prêt PRT-0002. Pourriez-vou…
  Client   : J'ai remboursé hier, le 15 novembre.
  Assistant: Je note que vous avez effectué votre remboursement le 15 novembre pour…
  Client   : 31 500 FCFA.
  Assistant: Je note le montant de votre remboursement qui s'élève à 31 500 FCFA po…
  Client   : Le paiement est passé par Lagune Money et j'ai reçu un SMS de confirma…
  Assistant: Je prends note que votre paiement de 31 500 FCFA a été effectué via La…
  Client   : Depuis, mon accès à un nouveau prêt est suspendu. C'est normal ?
  Assistant: Je prends note que votre accès à un nouveau prêt est suspendu suite à …
  Client   : Est-ce que des frais de retard vont m'être facturés ?
  Assistant: Je prends note de votre question concernant d'éventuels frais de retar…
  Client   : Combien de temps faut-il pour régulariser ?
  Assistant: Je prends note de votre question concernant le délai d

## <a id="bilan"></a>Bilan

In [8]:
for nom, run in RESULTATS.items():
    total = sum(run["tokens"]) + run["resume_tokens"]
    print(f"{nom:10s}: {total} tokens d'entrée au total (dont {run['resume_tokens']} pour les résumés), "
          f"{len(missing_elements(run['reponse']))} élément(s) manquant(s) sur 3")

complet   : 3571 tokens d'entrée au total (dont 0 pour les résumés), 0 élément(s) manquant(s) sur 3
troncature: 1849 tokens d'entrée au total (dont 0 pour les résumés), 3 élément(s) manquant(s) sur 3
résumé    : 3164 tokens d'entrée au total (dont 701 pour les résumés), 0 élément(s) manquant(s) sur 3


La troncature est gratuite et prévisible, mais elle oublie : la référence du prêt, donnée au premier message, n'est plus envoyée. Le résumé la conserve au prix d'un appel supplémentaire, et un détail peut s'y perdre ou s'y déformer : une question de contrôle, comme ici, permet de le vérifier. Le TP2 mesure les mêmes stratégies sur un agent avec tools.